Brianna Aceves | DSC360 | 8.2 Exercise

In [1]:
#Import Libraries
%pip install pandas numpy scikit-learn sklearn-crfsuite spacy

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install spacy-lookups-data

Note: you may need to restart the kernel to use updated packages.


In [3]:
#Install SpaCy english model
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------- ------------------------------ 3.1/12.8 MB 20.0 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 21.3 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 22.1 MB/s eta 0:00:01
     ----------------------------------- --- 11.5/12.8 MB 14.6 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 14.1 MB/s  0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
#Import libraries
import pandas as pd
import numpy as np
import spacy
import sklearn_crfsuite
import zipfile

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import metrics
from spacy import displacy
from IPython.display import display, Markdown

print("All libraries imported successfully.")

All libraries imported successfully.


In [5]:
import json

#Open the notebook
filename = "how-to-loading-and-fitting-dataset-to-scikit.ipynb"

with open(filename, "r", encoding="utf-8") as file:
    notebook = json.load(file)

#Display the code from Kaggle notebook
for i, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] == "code":

        print(f"\nCode Cell {i}:")
        print("".join(cell["source"])[:1200])


Code Cell 2:
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, f1_score

Code Cell 3:
dframe = pd.read_csv("../input/ner.csv", encoding = "ISO-8859-1", error_bad_lines=False)

Code Cell 5:
dframe.dropna(inplace=True)

Code Cell 7:
dframe[dframe.isnull().any(axis=1)].size

Code Cell 9:
dframe =  dframe[:5000]
x_df = dframe.drop(['Unnamed: 0', 'sentence_idx', 'tag'], axis=1)
x_df.head()

Code Cell 11:
vectorizer = DictVectorizer(sparse=False)
x = vectorizer.fit_transform(x_df.to_dict("records"))
x.shape

Code Cell 12:
y = dframe.tag.values
all_classes = np.unique(y)
all_classes.shape
y.shape

Code Cell 14:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
print(x_train.shape)
print(y_train.shape)

Code Cell 16:
clf = Perceptron(verbose=10,

I started by reviewing the Jupyter Notebook downloaded from Kaggle to understand how the original author loaded and prepared the NER dataset. Since the downloaded file was a Jupyter Notebook rather than a CSV, I checked its Python code to determine where the dataset was being loaded from. This helped me identify the data source needed to build my own NER model using Scikit-learn.

In [6]:
#Load the NER dataset
import pandas as pd

url = (
    "https://raw.githubusercontent.com/"
    "karndeepsingh/Named-Entity-Recognition/"
    "main/ner_dataset.csv"
)

df = pd.read_csv(url, encoding="latin1")

#View the first ten rows
display(df.head(10))

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O
5,NaN,through,IN,O
6,NaN,London,NNP,B-geo
7,NaN,to,TO,O
8,NaN,protest,VB,O
9,NaN,the,DT,O


After reviewing the Kaggle notebook, I loaded the actual NER dataset into a Pandas DataFrame. I displayed the first ten rows to get a better understanding of how the information was organized. The dataset contains individual words, their parts of speech, and assigned NER tags. These tags identify whether a word represents a person, location, organization, or another type of entity. Reviewing the dataset helped me understand what information I would be using to train my own NER model.

In [7]:
#Check the size of the dataset
print("Dataset Shape:", df.shape)

#Review the columns
print("\nColumn Names:")
print(df.columns.tolist())

#Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

#Review the NER tag distribution
print("\nNER Tag Distribution:")
print(df["Tag"].value_counts())

Dataset Shape: (1048575, 4)

Column Names:
['Sentence #', 'Word', 'POS', 'Tag']

Missing Values:
Sentence #    1000616
Word               10
POS                 0
Tag                 0
dtype: int64

NER Tag Distribution:
Tag
O        887908
B-geo     37644
B-tim     20333
B-org     20143
I-per     17251
B-per     16990
I-org     16784
B-gpe     15870
I-geo      7414
I-tim      6528
B-art       402
B-eve       308
I-art       297
I-eve       253
B-nat       201
I-gpe       198
I-nat        51
Name: count, dtype: int64


I reviewed the dataset's dimensions, column names, missing values, and the distribution of NER tags. The dataset contains different entity categories, including geographical locations, people, organizations, geopolitical entities, and time expressions. It also includes the O tag for words that are not named entities. I noticed that the words are stored in individual rows rather than complete sentences. This is important because the CRF model needs to consider the words surrounding an entity when making predictions. This initial exploration helped me understand how the dataset is structured and what preparation is needed before training the model.

In [8]:
#Fill in missing sentence identifiers
df["Sentence #"] = df["Sentence #"].ffill()

#Remove rows missing essential information
df = df.dropna(
    subset=["Sentence #", "Word", "POS", "Tag"]
).copy()

#Group words into complete sentences
sentences = []

for _, group in df.groupby("Sentence #", sort=False):

    sentence = list(
        zip(
            group["Word"].astype(str),
            group["POS"].astype(str),
            group["Tag"].astype(str)
        )
    )

    sentences.append(sentence)

print("Total Sentences:", len(sentences))

print("\nFirst Sentence:")
print(sentences[0])

Total Sentences: 47959

First Sentence:
[('Thousands', 'NNS', 'O'), ('of', 'IN', 'O'), ('demonstrators', 'NNS', 'O'), ('have', 'VBP', 'O'), ('marched', 'VBN', 'O'), ('through', 'IN', 'O'), ('London', 'NNP', 'B-geo'), ('to', 'TO', 'O'), ('protest', 'VB', 'O'), ('the', 'DT', 'O'), ('war', 'NN', 'O'), ('in', 'IN', 'O'), ('Iraq', 'NNP', 'B-geo'), ('and', 'CC', 'O'), ('demand', 'VB', 'O'), ('the', 'DT', 'O'), ('withdrawal', 'NN', 'O'), ('of', 'IN', 'O'), ('British', 'JJ', 'B-gpe'), ('troops', 'NNS', 'O'), ('from', 'IN', 'O'), ('that', 'DT', 'O'), ('country', 'NN', 'O'), ('.', '.', 'O')]


I prepared the dataset by filling in the missing sentence identifiers and removing rows that were missing information needed for training. After, I grouped the individual words into complete sentences while keeping their parts of speech and NER labels together. This step was important because the CRF model considers surrounding words when predicting entity labels. Keeping the sentences together allows the model to learn patterns based on context rather than looking at each word individually. Organizing the data this way also helps the model recognize entities made up of multiple words, such as a person's full name.

In [9]:
#Create features for each word
def word2features(sent, i):

    word = sent[i][0]
    postag = sent[i][1]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "postag": postag,
        "postag[:2]": postag[:2]
    }

    #Features for previous word
    if i > 0:

        prev_word = sent[i - 1][0]
        prev_pos = sent[i - 1][1]

        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:postag": prev_pos
        })

    else:
        features["BOS"] = True

    #Features for next word
    if i < len(sent) - 1:

        next_word = sent[i + 1][0]
        next_pos = sent[i + 1][1]

        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:postag": next_pos
        })

    else:
        features["EOS"] = True

    return features


#Convert sentences into feature dictionaries
def sent2features(sent):
    return [
        word2features(sent, i)
        for i in range(len(sent))
    ]


#Extract correct NER labels
def sent2labels(sent):
    return [
        tag for word, pos, tag in sent
    ]

print("Feature extraction functions created.")

Feature extraction functions created.


I created features that the CRF model will use to identify named entities. The features include whether a word is capitalized, whether it contains numbers, its ending letters, and its part of speech. I also included information about the words immediately before and after it. I think the surrounding words are especially important because the same word can mean different things depending on how it is used.

In [10]:
from sklearn.model_selection import train_test_split

#Divide the sentences into training and testing sets
train_sents, test_sents = train_test_split(
    sentences,
    test_size=0.20,
    random_state=42
)

#Create training features and labels
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

#Create testing features and labels
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

print("Training Sentences:", len(X_train))
print("Testing Sentences:", len(X_test))

Training Sentences: 38367
Testing Sentences: 9592


After creating the features, I divided the dataset into 80/20 testing data. This resulted in 38,367 sentences for training and 9,592 sentences for testing, giving me a total of 47,959 sentences. I used a random state of 42 so the split can be reproduced if I run the notebook again. I also split the data by complete sentences instead of individual words to preserve the surrounding context. Looking at the results, I think this is a reasonable split because it gives the model a large amount of data to learn from while still keeping almost 10,000 sentences available for evaluation. This should help me get a better idea of how well the model recognizes entities in new text.

In [15]:
#Train the CRF model with fewer iterations
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=50,
    all_possible_transitions=True,
    verbose=True
)

#Train the model
crf.fit(X_train, y_train)

print("NER model training completed successfully!")

loading training data to CRFsuite: 100%|██████████| 38367/38367 [00:09<00:00, 4218.38it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 137614
Seconds required: 2.029

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 50
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=3.88  loss=1404077.79 active=136672 feature_norm=1.00
Iter 2   time=3.52  loss=1274354.56 active=135464 feature_norm=5.56
Iter 3   time=1.77  loss=971578.02 active=129036 feature_norm=4.83
Iter 4   time=9.45  loss=922485.91 active=127481 feature_norm=4.04
Iter 5   time=1.90  loss=484273.83 active=136847 feature_norm=4.96
Iter 6   time=1.48  loss=344622.12 active=136582 feature_norm=5.49
Iter 7   time=1.72  loss=318981.94 active=135860 feature_norm=5.77
Iter 8   time=1.58  loss=270891.74 active=123275 feature_norm=7.09
Iter 9   time=1.81  loss=235210.85 active=117511 feature_norm

For this step, I trained my Named Entity Recognition model using a Conditional Random Field (CRF) classifier through sklearn-crfsuite. I used the 38,367 training sentences to help the model learn patterns between words, their parts of speech, and their assigned NER labels. Initially, I set the model to run for a maximum of 100 iterations. However, the training process was taking longer than expected, and Jupyter continued showing that the cell was running. At first, I thought the program might be stuck in an infinite loop, but I learned that training a CRF model on a large dataset can take a considerable amount of processing time. To make the training process more manageable, I reduced the maximum number of iterations from 100 to 50. I also enabled the verbose setting so I could monitor the training progress instead of only seeing a running cell. After making these adjustments, the model completed training successfully. This experience helped me understand how model complexity, dataset size, and training settings can affect processing time. It also showed me why monitoring the training process is important when working with larger datasets.

In [16]:
#Evaluate model performance
from sklearn_crfsuite import metrics

#Predict NER labels using the testing data
y_pred = crf.predict(X_test)

#Exclude the O label
labels = [
    label for label in crf.classes_
    if label != "O"
]

#Calculate weighted F1 score
f1 = metrics.flat_f1_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels,
    zero_division=0
)

print("Weighted F1 Score:", round(f1, 3))

#Display classification report
print("\nClassification Report:")

print(
    metrics.flat_classification_report(
        y_test,
        y_pred,
        labels=sorted(labels),
        digits=3,
        zero_division=0
    )
)

Weighted F1 Score: 0.852

Classification Report:
              precision    recall  f1-score   support

       B-art      0.406     0.151     0.220        86
       B-eve      0.471     0.267     0.340        60
       B-geo      0.861     0.907     0.883      7664
       B-gpe      0.965     0.941     0.953      3175
       B-nat      0.682     0.300     0.417        50
       B-org      0.808     0.740     0.773      3913
       B-per      0.841     0.826     0.833      3389
       B-tim      0.933     0.873     0.902      4049
       I-art      0.200     0.034     0.059        58
       I-eve      0.400     0.226     0.289        53
       I-geo      0.808     0.797     0.802      1450
       I-gpe      0.862     0.641     0.735        39
       I-nat      0.333     0.167     0.222        12
       I-org      0.841     0.792     0.816      3315
       I-per      0.853     0.897     0.874      3445
       I-tim      0.845     0.744     0.791      1300

   micro avg      0.864     0.8

After training my CRF model, I evaluated its performance using the 9,592 testing sentences. The model achieved a weighted F1 score of 0.852 (85.2%), which shows that it performed pretty well overall. Its precision was 0.861, and recall was 0.845. Looking at the results, the model performed best with geopolitical entities (B-gpe), which had an F1 score of 0.953, followed by time expressions (B-tim) at 0.902. However, it struggled with less common categories, especially I-art, which scored only 0.059. I think these results show that the model learned to recognize common entities fairly well but had more difficulty with categories that had fewer examples. Since I reduced the training iterations from 100 to 50, there may still be room for improvement with additional training.

In [17]:
#Analyze the strongest and weakest NER labels
from sklearn.metrics import classification_report

#Flatten actual and predicted labels
true_flat = [
    tag for sentence in y_test
    for tag in sentence
]

pred_flat = [
    tag for sentence in y_pred
    for tag in sentence
]

#Generate classification report
report = classification_report(
    true_flat,
    pred_flat,
    labels=sorted(labels),
    output_dict=True,
    zero_division=0
)

#Convert results into a DataFrame
report_df = pd.DataFrame(report).T

#Keep individual entity labels
tag_scores = report_df.loc[sorted(labels)]

#Sort by F1 score
tag_scores = tag_scores.sort_values(
    by="f1-score",
    ascending=False
)

display(tag_scores.round(3))

print("Highest F1 Label:", tag_scores.index[0])
print("Lowest F1 Label:", tag_scores.index[-1])

,precision,recall,f1-score,support
B-gpe,0.965,0.941,0.953,3175.0
B-tim,0.933,0.873,0.902,4049.0
B-geo,0.861,0.907,0.883,7664.0
I-per,0.853,0.897,0.874,3445.0
B-per,0.841,0.826,0.833,3389.0
I-org,0.841,0.792,0.816,3315.0
I-geo,0.808,0.797,0.802,1450.0
I-tim,0.845,0.744,0.791,1300.0
B-org,0.808,0.740,0.773,3913.0
I-gpe,0.862,0.641,0.735,39.0


Highest F1 Label: B-gpe
Lowest F1 Label: I-art


After reviewing the classification report, I compared the F1 scores to see which NER labels my model recognized best. The highest-performing label was B-gpe, with an F1 score of 0.953, followed by B-tim at 0.902 and B-geo at 0.883. This shows that the model did pretty well at recognizing common locations and time-related entities. On the other hand, the model struggled with I-art, which had the lowest F1 score of 0.059. I noticed that categories with fewer examples generally had lower scores, which may explain why some labels were harder to predict. Overall, these results show that the model learned common entity patterns fairly well but still has room for improvement with less frequent categories.

In [18]:
#Test NER Model
import spacy

#Load spaCy's English language model
nlp = spacy.load("en_core_web_sm")

# Required assignment sentence
test_sentence = (
    "Fourteen days ago, Emperor Palpatine left "
    "San Diego, CA for Tatooine to follow Luke Skywalker."
)

#Get tokens and POS tags without spaCy NER
with nlp.select_pipes(disable=["ner"]):
    doc_for_crf = nlp(test_sentence)

#Prepare sentence for the custom CRF model
custom_sentence = [
    (token.text, token.tag_, "O")
    for token in doc_for_crf
]

#Create word features
test_features = sent2features(custom_sentence)

#Predict NER tags
custom_predictions = crf.predict_single(test_features)

#Display the predictions
custom_results = pd.DataFrame({
    "Word": [
        word for word, pos, tag in custom_sentence
    ],
    "Predicted Tag": custom_predictions
})

print("Custom CRF Model Results:")
display(custom_results)

Custom CRF Model Results:


,Word,Predicted Tag
0,Fourteen,O
1,days,O
2,ago,O
3,",",O
4,Emperor,B-per
5,Palpatine,I-per
6,left,O
7,San,B-geo
8,Diego,I-geo
9,",",O


After training my CRF model, I tested it using the required sentence to see how well it recognized named entities. The model correctly identified Emperor Palpatine and Luke Skywalker as people (B-per and I-per). It also recognized San Diego as a geographical location (B-geo and I-geo). However, it did not recognize "Fourteen days ago" as a time expression and labeled those words as O. One interesting mistake was how the model classified CA as the beginning of an organization (B-org), then included "for Tatooine" as part of that organization (I-org). This shows that the model had difficulty recognizing the fictional location and determining where the entity should end. Overall, I think the model performed fairly well with familiar naming patterns, but it struggled with unfamiliar locations and certain sentence structures. This helped me see how context and training data can affect NER predictions.

In [19]:
#Run NER Engine
#Test the sentence using spaCy's NER model
from spacy import displacy

#Process the required sentence
doc_spacy = nlp(test_sentence)

#Extract recognized entities
spacy_results = pd.DataFrame(
    [
        {
            "Entity": ent.text,
            "NER Label": ent.label_,
            "Description": spacy.explain(ent.label_)
        }
        for ent in doc_spacy.ents
    ],
    columns=["Entity", "NER Label", "Description"]
)

print("spaCy NER Results:")
display(spacy_results)

#Highlight the entities in the sentence
displacy.render(
    doc_spacy,
    style="ent",
    jupyter=True
)

spaCy NER Results:


,Entity,NER Label,Description
0,Fourteen days ago,DATE,Absolute or relative dates or periods
1,Palpatine,PERSON,"People, including fictional"
2,San Diego,GPE,"Countries, cities, states"
3,CA for Tatooine,WORK_OF_ART,"Titles of books, songs, etc."
4,Luke Skywalker,PERSON,"People, including fictional"


I tested the same sentence using spaCy's pretrained NER model to see how its results compared to my custom CRF model. SpaCy correctly identified "Fourteen days ago" as a date (Date), "San Diego" as a geopolitical location (GPE), and "Luke Skywalker" as a person (Person). However, it only recognized "Palpatine" as a person and left out "Emperor." One interesting mistake was that spaCy classified "CA for Tatooine" as a work of art (WORK_OF_ART), rather than recognizing CA and Tatooine as locations. My custom model also struggled with this part of the sentence, although it incorrectly labeled those words as an organization instead. Overall, spaCy did better at recognizing the time expression, but both models had trouble with certain names and locations. This shows me that even pretrained models can make mistakes when interpreting unfamiliar text.

In [20]:
#Compare both models word by word
comparison = pd.DataFrame({
    "Word": [token.text for token in doc_spacy],
    "Custom CRF Tag": custom_predictions,
    "spaCy Tag": [
        (
            token.ent_iob_ + "-" + token.ent_type_
            if token.ent_iob_ != "O"
            else "O"
        )
        for token in doc_spacy
    ]
})

print("Custom CRF vs. spaCy Comparison:")
display(comparison)

Custom CRF vs. spaCy Comparison:


,Word,Custom CRF Tag,spaCy Tag
0,Fourteen,O,B-DATE
1,days,O,I-DATE
2,ago,O,I-DATE
3,",",O,O
4,Emperor,B-per,O
5,Palpatine,I-per,B-PERSON
6,left,O,O
7,San,B-geo,B-GPE
8,Diego,I-geo,I-GPE
9,",",O,O


After running the same sentence through both models, I compared their predictions side by side to see where they agreed and differed. Both models correctly recognized San Diego as a location and Luke Skywalker as a person. My CRF model also identified the full name Emperor Palpatine, while spaCy only recognized Palpatine. However, spaCy performed better with the time expression "Fourteen days ago," correctly labeling it as a date, while my model missed it completely. Interestingly enough, both models struggled with "CA for Tatooine." My CRF model incorrectly classified it as an organization, while spaCy labeled it as a work of art. Neither model correctly recognized Tatooine as a location. I think the comparison shows that both models have strengths and weaknesses. It also helped me understand how training data and word context can influence NER predictions, especially when working with unfamiliar names or locations.

In [21]:
#Convert BIO tags into complete entities
def extract_entities(words, tags):

    entities = []
    current_words = []
    current_type = None

    def save_entity():
        if current_words:
            entities.append({
                "Entity": " ".join(current_words),
                "Tag": current_type
            })

    for word, tag in zip(words, tags):

        if tag == "O":
            save_entity()
            current_words = []
            current_type = None
            continue

        if "-" not in tag:
            continue

        prefix, entity_type = tag.split("-", 1)

        if (
            prefix == "B"
            or entity_type != current_type
            or not current_words
        ):
            save_entity()
            current_words = [word]
            current_type = entity_type

        elif prefix == "I":
            current_words.append(word)

    save_entity()

    return pd.DataFrame(
        entities,
        columns=["Entity", "Tag"]
    )


#Extract entities from custom CRF model
crf_entities = extract_entities(
    custom_results["Word"].tolist(),
    custom_predictions
)

print("Question 1: Custom CRF Results")
display(crf_entities)

print("\nQuestion 2: spaCy Results")
display(spacy_results)

#Compare complete entity text
crf_names = set(
    crf_entities["Entity"].str.lower()
)

spacy_names = set(
    spacy_results["Entity"].str.lower()
)

print("\nEntities recognized by both:")
print(sorted(crf_names & spacy_names))

print("\nEntities recognized only by CRF:")
print(sorted(crf_names - spacy_names))

print("\nEntities recognized only by spaCy:")
print(sorted(spacy_names - crf_names))

Question 1: Custom CRF Results


,Entity,Tag
0,Emperor Palpatine,per
1,San Diego,geo
2,CA for Tatooine,org
3,Luke Skywalker,per



Question 2: spaCy Results


,Entity,NER Label,Description
0,Fourteen days ago,DATE,Absolute or relative dates or periods
1,Palpatine,PERSON,"People, including fictional"
2,San Diego,GPE,"Countries, cities, states"
3,CA for Tatooine,WORK_OF_ART,"Titles of books, songs, etc."
4,Luke Skywalker,PERSON,"People, including fictional"



Entities recognized by both:
['ca for tatooine', 'luke skywalker', 'san diego']

Entities recognized only by CRF:
['emperor palpatine']

Entities recognized only by spaCy:
['fourteen days ago', 'palpatine']


After comparing the complete entities, I found that both models recognized San Diego and Luke Skywalker correctly. My custom CRF model also identified the full name Emperor Palpatine, while spaCy only recognized Palpatine. SpaCy performed better with the time expression "Fourteen days ago," correctly labeling it as a date, while my model missed it completely. Both models also identified "CA for Tatooine" as an entity, but neither classified it correctly. My CRF model labeled it as an organization (org), while spaCy classified it as a work of art (WORK_OF_ART). This was interesting because both models grouped the same words together but interpreted them differently. Overall, the comparison showed me that both models have strengths and weaknesses. My custom model did better at recognizing the complete character name, while spaCy handled the time expression more accurately. This really showed me how training data and context can affect NER predictions.